# Regular

It is a set of ClickHouse functions whose results depends on only one row.

## arrayReduce

The `arrayReduce` function applies an aggregation function to the given array. The syntax is `arrayReduce(func, arr1[, arr2, arr3, ... arrN])`.

Imporant to note:

- If the aggreagation function takes several arguments, pass each one in the corresponding array.
- If the aggreagation function has a scalar parameteriaztion, specify the parameter as usual: `function(<scalar parameter>)`.

---

The following cell applies `max` function to get given array.

In [1]:
--ClickHouse
SELECT arrayReduce('max', [1, 2, 3]);

"arrayReduce('max', [1, 2, 3])"
3


The following example uses the aggregation functions `maxIf` and `argMax`, which require two arrays:

In [3]:
--ClickHouse
SELECT
    arrayReduce('maxIf', [1, 2, 3], [1, 1, 0]) AS max_if,
    arrayReduce('argMax', ['a', 'b', 'c'], [10, 20, 30]) AS arg_max
;

max_if,arg_max
2,c


The histogram function is an aggregation function that requires the scalar parametrization. The following cell show how it is supposed to be used with `arrayReduce`:

In [4]:
--ClickHouse
SELECT arrayReduce('histogram(3)', [1, 2, 3, 4, 5, 6, 7, 8, 9]);

"arrayReduce('histogram(3)', [1, 2, 3, 4, 5, 6, 7, 8, 9])"
"[(1.0, 4.0, 3.75), (4.0, 6.75, 2.375), (6.75, 9.0, 2.875)]"


## arrayFilter

The array filter function returns the elements of the passed array for which given lambda function returns true, thus you can filter the elements of the array.

Check the reference in the [arrayFilter](https://clickhouse.com/docs/sql-reference/functions/array-functions#arrayFilter) section of the reference.

---

The following cell uses the `filterArray` function to filter the elements of the `numbers` array that correspond to the `'valid'` value in the `marker` array.

In [12]:
--ClickHouse
SELECT 
    [0, 1, 2, 3, 4] AS numbers,
    ['valid', 'invalid', 'valid', 'some other', 'valid'] AS markers,
    arrayFilter(
    (x, marker) -> (marker = 'valid'), numbers, markers
    ) AS fun_out
;

numbers,markers,fun_out
"[0, 1, 2, 3, 4]","['valid', 'invalid', 'valid', 'some other', 'valid']","[0, 2, 4]"


## arrayJoin

The `arrayJoin` function generates a row for each element of the given array. The values of the other columns corresponding to the array are duplicated for each element of the joined array.

---

Consider few examples of the `arrayJoin` usage.

The following cell just unfolds single array:

In [3]:
--ClickHouse
SELECT
    arrayJoin(['a', 'b', 'c']),
    'ancor' as ancor;

"arrayJoin(['a', 'b', 'c'])",ancor
a,ancor
b,ancor
c,ancor


Note that the value of the "ancor" column is replicated for each element of the original array.

The following cell is a close example. A corresponding set of lines is generated for each array in the original column:

In [13]:
--ClickHouse
WITH
arr_join_example AS (
    SELECT * FROM
        VALUES (
            'my_arrays Array(TEXT), category TEXT',
            (['a', 'b', 'c'], 'one'),
            (['x', 'y'], 'two')
        )
)
SELECT arrayJoin(my_arrays), category FROM arr_join_example;

arrayJoin(my_arrays),category
a,one
b,one
c,one
x,two
y,two


The next cell is a slightly trickier example, with two `arrayJoin` functions in the same query:

In [5]:
--ClickHouse
SELECT
    arrayJoin(['a', 'b', 'c']),
    arrayJoin(['x', 'y', 'z']);

"arrayJoin(['a', 'b', 'c'])","arrayJoin(['x', 'y', 'z'])"
a,x
a,y
a,z
b,x
b,y
b,z
c,x
c,y
c,z


The `['a', 'b', 'c']` array is unfolded first - each element is paired with a copy of `['x', 'y', 'z']`. Then `['x', 'y', 'z']` is unfolded for each of `a`, `b`, and `c`, producing all combinations of the original arrays.